# Week 7 · Diffusion, part 2: generate images, then generate them faster

In Session 1 you built the forward process, the denoising loss, and a U-Net
that knows the noise level: activities 1 to 4 of `diffusion.ipynb`. Today you
continue from there.

5. **Train** the network across all noise levels.
6. **Generate:** build the DDPM reverse step, test it, and diagnose flawed samples.
7. **Judge** the generator.
8. **Try** a tempting shortcut that generates with fewer network evaluations,
   and locate its error.
9. **Understand** what a correct jump over timesteps looks like: the DDIM step.
10. **Experiment:** compare 64, 16 and 4 network evaluations.
11. Individual exit ticket.

Activities 12 and 13 are extended exercises, for fast pairs and for home.

Complete the **TODOs** and the written questions. Predict before running; rerun
cells in order after edits. Switch driver after activity 7.

You do not need your notebook from Session 1: its parts are supplied below,
completed. Uses NumPy, Matplotlib, and CPU PyTorch; no GPU or download needed.
All training takes under a minute. The notebook follows Lesson 7 and Lab 7 of
the course book.

In [ ]:
import math
import time
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(2)
torch.manual_seed(7)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})
BLUE, ORANGE, GREEN, PURPLE = "#0072B2", "#D55E00", "#009E73", "#AA4499"  # colour-blind-safe
GRAY = "#6b6b6b"
print("Ready: PyTorch", torch.__version__, "| CPU")

## Before you start: the parts from Session 1

The next three cells hold what you built in activities 1 to 4, completed. Run
them. Compare them with your own code if you like.

| Activity | Function | What it does |
|---|---|---|
| 1 | `make_disks` | 16×16 images of one bright disk, values in $[-1,1]$ |
| 2 | `cosine_schedule` | signal levels $\bar\alpha_t$ for $t=0,\dots,64$, stored in `schedule.abar[t]` |
| 2 | `q_sample` | $x_t=\sqrt{\bar\alpha_t}\;x_0+\sqrt{1-\bar\alpha_t}\;\epsilon$ |
| 3 | `predict_clean` | $\hat x_0=\big(x_t-\sqrt{1-\bar\alpha_t}\;\epsilon_\theta\big)\,/\sqrt{\bar\alpha_t}$ |
| 3 | `diffusion_loss` | mean squared error between the predicted and the true noise |
| 3 | `ZeroNoise`, `CopyInput`, `Oracle` | reference denoisers: two trivial ones and a perfect one |
| 4 | `TinyTimeUNet` | predicts the noise in $x_t$ from $x_t$ and the timestep $t$ |

In [ ]:
# Supplied (Session 1, activities 1 to 3): data, plotting, schedule, forward process, loss.
def make_disks(n, seed):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:16, :16]
    images = np.zeros((n, 1, 16, 16), dtype=np.float32)
    for i in range(n):
        radius = rng.uniform(2.0, 4.0)
        cy, cx = rng.uniform(radius + 1, 14 - radius, size=2)  # the whole disk stays inside
        distance = np.sqrt((yy - cy)**2 + (xx - cx)**2)
        images[i, 0] = 1 / (1 + np.exp((distance - radius) / 0.6))  # soft edge, values in (0, 1)
    return torch.tensor(2 * images - 1)  # rescale brightness from [0, 1] to [-1, 1]

# Show rows of images. `rows` is a list of tensors with shape (K, 1, 16, 16).
def show_rows(rows, labels=None, titles=None, suptitle=None, size=1.0):
    columns = max(len(row) for row in rows)
    width = size * columns + (1.1 if labels else 0.2)
    fig, axes = plt.subplots(len(rows), columns, squeeze=False,
                             figsize=(width, 1.08 * size * len(rows) + 0.5))
    for ax in axes.flat:
        ax.set(xticks=[], yticks=[])
        ax.spines[:].set_visible(False)
    for r, images in enumerate(rows):
        for c, image in enumerate(images):
            axes[r, c].imshow(image[0].detach().numpy(), cmap="gray", vmin=-1, vmax=1,
                              interpolation="nearest")
        if labels:
            axes[r, 0].set_ylabel(labels[r], rotation=0, ha="right", va="center", fontsize=9)
    for ax, title in zip(axes[0], titles or []):
        ax.set_title(title, fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    plt.show()

# Index 0 is a placeholder, so schedule.abar[t] matches the maths for t = 0, ..., T.
def make_schedule(beta):
    alpha = 1 - beta
    abar = torch.cumprod(alpha, dim=0)  # abar[0] = 1
    return SimpleNamespace(steps=len(beta) - 1, beta=beta.float(), alpha=alpha.float(),
                           abar=abar.float())

def cosine_schedule(steps, offset=0.008):
    t = torch.arange(steps + 1, dtype=torch.float64)
    f = torch.cos((t / steps + offset) / (1 + offset) * math.pi / 2)**2
    beta = torch.zeros(steps + 1, dtype=torch.float64)
    beta[1:] = (1 - f[1:] / f[:-1]).clamp(max=0.999)  # beta[t] = 1 - abar[t] / abar[t-1]
    return make_schedule(beta)

def q_sample(clean, t, noise, schedule):
    # Direct noising: jump from clean images x_0 to x_t. `t` holds one timestep per image.
    abar = schedule.abar[t].view(-1, 1, 1, 1)
    return abar.sqrt() * clean + (1 - abar).sqrt() * noise

def predict_clean(noisy, t, noise, schedule):
    # Turn a noise prediction for x_t into an estimate of the clean image x_0.
    abar = schedule.abar[t].view(-1, 1, 1, 1)
    return (noisy - (1 - abar).sqrt() * noise) / abar.sqrt()

def diffusion_loss(model, clean, schedule, generator):
    # Noise-prediction loss for one minibatch: a random timestep and fresh noise for every image.
    t = torch.randint(1, schedule.steps + 1, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    return F.mse_loss(model(q_sample(clean, t, noise, schedule), t), noise)

train = make_disks(256, seed=7)  # used for training
valid = make_disks(64, seed=8)   # independent images, never used for training
T = 64
schedule = cosine_schedule(T)
print("Signal level abar[t] at t = 0, 16, 32, 48, 64:",
      "  ".join(f"{schedule.abar[t].item():.4f}" for t in (0, 16, 32, 48, 64)))

In [ ]:
# Supplied (Session 1, activity 4): the time-aware U-Net. Below it: the training loop.
def time_features(t, steps):
    tau = t.float() / steps
    features = [tau, torch.sin(math.pi * tau), torch.cos(math.pi * tau),
                torch.sin(2 * math.pi * tau), torch.cos(2 * math.pi * tau)]
    return torch.stack(features, dim=1)  # (N, 5)

class TimeBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(4, out_channels)
        self.time = nn.Linear(32, out_channels)  # time embedding -> one shift per channel
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(4, out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, 1)

    def forward(self, x, emb):
        h = self.norm1(self.conv1(x))             # (N, C, H, W)
        h = h + self.time(emb)[:, :, None, None]  # the time shift, one value per channel
        h = F.silu(h)
        h = self.norm2(self.conv2(h))
        return F.silu(h + self.shortcut(x))       # residual shortcut

class TinyTimeUNet(nn.Module):
    def __init__(self, steps, width=8):
        super().__init__()
        self.steps = steps
        self.embedding = nn.Sequential(nn.Linear(5, 32), nn.SiLU(), nn.Linear(32, 32))
        self.enc1 = TimeBlock(1, width)
        self.enc2 = TimeBlock(width, 2 * width)
        self.bottom = TimeBlock(2 * width, 4 * width)
        self.dec2 = TimeBlock(4 * width + 2 * width, 2 * width)
        self.dec1 = TimeBlock(2 * width + width, width)
        self.output = nn.Conv2d(width, 1, kernel_size=1)

    def forward(self, x, t):
        emb = self.embedding(time_features(t, self.steps))    # (N, 32)
        return self.unet(x, emb)

    def unet(self, x, emb):
        skip1 = self.enc1(x, emb)                             # 8 × 16 × 16
        skip2 = self.enc2(F.avg_pool2d(skip1, 2), emb)        # 16 × 8 × 8
        h = self.bottom(F.avg_pool2d(skip2, 2), emb)          # 32 × 4 × 4
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.dec2(torch.cat([h, skip2], dim=1), emb)      # 16 × 8 × 8
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.dec1(torch.cat([h, skip1], dim=1), emb)      # 8 × 16 × 16
        return self.output(h)                                 # predicted noise: no sigmoid

def train_updates(model, optimizer, images, schedule, updates, generator):
    model.train()
    losses = []
    for _ in range(updates):
        batch = images[torch.randint(len(images), (32,), generator=generator)]  # 32 random images
        loss = diffusion_loss(model, batch, schedule, generator)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

In [ ]:
# Supplied (Session 1, activity 3): reference denoisers. They are called like the network.
class ZeroNoise(nn.Module):  # "there is no noise"
    def forward(self, x, t):
        return torch.zeros_like(x)

class CopyInput(nn.Module):  # "everything is noise"
    def forward(self, x, t):
        return x

class Oracle(nn.Module):
    # A perfect denoiser for one known clean image. It also keeps a logbook of its calls,
    # which activity 8 will read.
    def __init__(self, clean, schedule):
        super().__init__()
        self.clean, self.schedule, self.log = clean, schedule, []

    def noise_in(self, x):
        # Explain as much of x as possible by the clean image. What remains is noise.
        weight = (x * self.clean).mean() / (self.clean**2).mean()
        return (x - weight * self.clean).std().item()

    def forward(self, x, t):
        self.log.append((t[0].item(), self.noise_in(x)))  # the label, the noise in the state
        abar = self.schedule.abar[t].view(-1, 1, 1, 1)
        # The noise prediction for which predict_clean returns exactly the known image.
        return (x - abar.sqrt() * self.clean) / (1 - abar).sqrt()

print("Parameters of the network:", sum(p.numel() for p in TinyTimeUNet(T).parameters()))

### Recall

Answer from memory, each for yourself, then compare with your partner.

1. What does the network receive, and what does it predict?
2. Where does its training target come from?
3. Why is the network told the timestep?

_Your answers:_ …

## 5. Train across all noise levels

One update: draw 32 training images, give each its own random timestep and
noise, predict the noise, and take an Adam step on your `diffusion_loss`. Start
with a **short run of 200 updates**.

To see what the network has learned, the cell measures the validation error
**separately at every noise level** and compares it with the two baselines
from activity 3.

**Predict before running:** at which noise levels will the network have its
lowest error, and at which its highest?

_Your prediction:_ …

In [ ]:
# Supplied evaluation helpers.
@torch.no_grad()
def denoising_errors(model, images, schedule):
    # Mean squared noise error at every timestep 1, ..., T.
    # Every model is tested with the same noise.
    model.eval()
    noise = torch.randn(images.shape, generator=torch.Generator().manual_seed(0))
    errors = []
    for step in range(1, schedule.steps + 1):
        t = torch.full((len(images),), step)
        prediction = model(q_sample(images, t, noise, schedule), t)
        errors.append(F.mse_loss(prediction, noise).item())
    return np.array(errors)

def plot_errors(errors):
    fig, ax = plt.subplots(figsize=(6, 3.2))
    for name, color in zip(errors, [GRAY, ORANGE, BLUE, GREEN]):
        ax.plot(np.arange(1, T + 1), errors[name], color=color, label=name)
    ax.set(xlabel="Timestep $t$   (almost clean → almost pure noise)",
           ylabel="Validation noise error (log scale)", yscale="log", xlim=(1, T), ylim=(5e-4, 3))
    ax.grid(color="#e6e6e6", linewidth=0.8)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    plt.show()

report_steps = [1, 16, 32, 48, 64]

def print_errors(errors):
    print(f"{'Validation noise error at t =':30s}" + "".join(f"{t:>9d}" for t in report_steps))
    for name, values in errors.items():
        print(f"{name:30s}" + "".join(f"{values[t - 1]:9.4f}" for t in report_steps))

torch.manual_seed(17)  # the same initial weights on every run of this cell
model = TinyTimeUNet(T)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
train_generator = torch.Generator().manual_seed(19)
losses = train_updates(model, optimizer, train, schedule, 200, train_generator)

fig, ax = plt.subplots(figsize=(6, 2.6))
ax.plot(np.arange(1, len(losses) + 1), losses, color=BLUE, linewidth=1)
ax.set(xlabel="Update", ylabel="Training loss (log scale)", yscale="log", xlim=(1, len(losses)))
ax.grid(color="#e6e6e6", linewidth=0.8)
fig.tight_layout()
plt.show()

errors = {"Always zero": denoising_errors(ZeroNoise(), valid, schedule),
          "Copy the input": denoising_errors(CopyInput(), valid, schedule),
          f"Network, {len(losses)} updates": denoising_errors(model, valid, schedule)}
plot_errors(errors)
print_errors(errors)
factor = ((1 - schedule.abar) / schedule.abar).sqrt()
print(f"{'Noise error → image error: ×':30s}"
      + "".join(f"{factor[t].item():9.3f}" for t in report_steps))

The last printed row is the factor $\sqrt{1-\bar\alpha_t}\,/\sqrt{\bar\alpha_t}$:
an error of size $\delta$ in the predicted noise changes the clean estimate
$\hat x_0$ by $\delta$ times this factor.

**Stop and interpret:**

1. Does the network beat *both* baselines at every noise level? Look closely
   at the right-hand end of the plot.
2. The network's error is smallest near $t=64$. Is that an achievement? What
   does *copy the input* reach there, and why?
3. The error is largest near $t=1$, where the image is almost clean. Use the
   factor to compare: at which end of the chain do small noise errors become
   large errors in the clean estimate?
4. The training loss jumps up and down far more than in Week 6. Name one
   source of randomness in every update that Week 6 did not have.

_Your observations:_ …

## 6. The reverse process: generate in small steps

Sampling runs the chain backwards. Start from pure noise
$x_T\sim\mathcal N(0,I)$ and go from $x_t$ to $x_{t-1}$ until you reach $x_0$.

**If we knew the clean image**, the previous state would be easy to draw. Given
$x_t$ and $x_0$, it is Gaussian with mean $\tilde\mu_t$ and variance $\tilde\beta_t$:

$$\tilde\mu_t(x_t,x_0)=\underbrace{\frac{\beta_t\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_t}}_{\text{clean weight}}\;x_0\;+\;\underbrace{\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}}_{\text{state weight}}\;x_t,\qquad \tilde\beta_t=\beta_t\,\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}.$$

**We do not know the clean image**, so the network supplies its current
estimate $\hat x_0$ from activity 3, clipped to the valid range $[-1,1]$. One
**DDPM step** is

$$x_{t-1}=\tilde\mu_t(x_t,\hat x_0)+\sqrt{\tilde\beta_t}\;z,\qquad z\sim\mathcal N(0,I).$$

The step mixes the clean estimate with the current state and adds **fresh
noise**. Then the network looks again, at a slightly lower noise level.

**Pen and paper:** evaluate the clean weight, the state weight and
$\tilde\beta_t$ for the last step, $t=1$. Use $\bar\alpha_0=1$ and
$\bar\alpha_1=\alpha_1=1-\beta_1$. What does the last step return? Does it add
noise?

_Your calculation:_ …

In [ ]:
# Supplied: the three reverse-step tables, indexed by t like schedule.abar. Index 0 is unused.
beta, alpha, abar = schedule.beta[1:], schedule.alpha[1:], schedule.abar[1:]
previous = schedule.abar[:-1]  # abar[t-1] for t = 1, ..., T; begins with abar[0] = 1
unused = torch.zeros(1)
schedule.clean_weight = torch.cat([unused, beta * previous.sqrt() / (1 - abar)])
schedule.state_weight = torch.cat([unused, alpha.sqrt() * (1 - previous) / (1 - abar)])
schedule.variance = torch.cat([unused, beta * (1 - previous) / (1 - abar)])

fig, ax = plt.subplots(figsize=(6, 3))
steps = np.arange(1, T + 1)
ax.plot(steps, schedule.state_weight[1:].numpy(), color=BLUE, label="State weight")
ax.plot(steps, schedule.clean_weight[1:].numpy(), color=ORANGE, label="Clean weight")
ax.plot(steps, schedule.variance[1:].sqrt().numpy(), color=GREEN,
        label=r"Fresh noise $\sqrt{\tilde\beta_t}$")
ax.set(xlabel="Timestep $t$   (sampling runs from right to left)",
       ylabel="Weight or standard deviation", xlim=(1, T), ylim=(0, 1.05))
ax.grid(color="#e6e6e6", linewidth=0.8)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()
for t in (64, 32, 1):
    print(f"t = {t:2d}: clean weight {schedule.clean_weight[t].item():.3f} | "
          f"state weight {schedule.state_weight[t].item():.3f} | "
          f"fresh noise {schedule.variance[t].sqrt().item():.3f}")

**Read the plot** and compare the printed values at $t=1$ with your calculation.
During most of the chain, is the new state closer to the old state or to the
clean estimate? Where does the clean estimate take over?

_Your reading:_ …

In [ ]:
@torch.no_grad()
def ddpm_step(model, x, step, noise, schedule):
    # One reverse step from x_t (`x`) to x_{t-1}. `step` is the integer t, `noise` is z.
    t = torch.full((len(x),), step)
    clean = predict_clean(x, t, model(x, t), schedule).clamp(-1, 1)  # clipped estimate of x_0
    # TODO 5a: mean of the previous state. Combine `clean` and `x` with
    # schedule.clean_weight[step] and schedule.state_weight[step].
    mean = None
    if mean is None:
        return None
    # TODO 5b: return the mean plus `noise` with standard deviation sqrt(schedule.variance[step]).
    return None

# Supplied: run all reverse steps T, ..., 1.
# Returns the results and the states x_t for the timesteps listed in `record`.
@torch.no_grad()
def sample_ddpm(model, initial, schedule, seed, fresh_noise=True, record=()):
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x, states = initial.clone(), {}
    for step in range(schedule.steps, 0, -1):
        if step in record:
            states[step] = x.clone()
        z = torch.randn(x.shape, generator=generator)
        x = ddpm_step(model, x, step, z if fresh_noise else torch.zeros_like(z), schedule)
    return x, states

known = valid[:1]  # the oracle knows this clean image and leads every clean estimate to it
oracle = Oracle(known, schedule)
starts = torch.randn(8, 1, 16, 16, generator=torch.Generator().manual_seed(6))
no_noise = torch.zeros_like(starts)
sampler_ready = ddpm_step(oracle, starts, T, no_noise, schedule) is not None
if sampler_ready:
    # 1. One step without fresh noise must match an equivalent formula for the same mean.
    state = q_sample(known.repeat(8, 1, 1, 1), torch.full((8,), 32), starts, schedule)
    same_mean = state - schedule.beta[32] / (1 - schedule.abar[32]).sqrt() * starts
    same_mean = same_mean / schedule.alpha[32].sqrt()
    step_mean = ddpm_step(oracle, state, 32, no_noise, schedule)
    mean_ok = torch.allclose(step_mean, same_mean, atol=1e-5)
    assert mean_ok, "TODO 5a: the mean is wrong. Check which weight multiplies which image."
    # 2. Fresh noise enters with standard deviation sqrt(variance): none at all at t = 1.
    for step in (32, 1):
        with_noise = ddpm_step(oracle, starts, step, torch.ones_like(starts), schedule)
        without = ddpm_step(oracle, starts, step, no_noise, schedule)
        expected = torch.full_like(starts, schedule.variance[step].sqrt().item())
        noise_ok = torch.allclose(with_noise - without, expected, atol=1e-5)
        assert noise_ok, "TODO 5b: the noise needs standard deviation sqrt(variance[step])."
    # 3. With exact noise predictions, every start must end at the known image.
    result, _ = sample_ddpm(oracle, starts, schedule, seed=7)
    deviation = (result - known).abs().max().item()
    assert deviation < 1e-3, f"All starts must end at the known image (off by {deviation:.2f})."
    print("Sampler checked: 8 different starts all end at the known image "
          f"(largest deviation {deviation:.0e}).")
else:
    print("Complete TODO 5, then rerun this cell.")

**Explain:** the oracle test replaces the network with a perfect denoiser for
one known image. What does a passing test tell you about your sampler? What can
it *not* tell you about your trained network?

_Your answer:_ …

### Generate

Draw 64 starting-noise images and run all 64 reverse steps with the network from
the short training run. The first figure shows 32 results, numbered row by row
from 0. The second follows the samples listed in `watch` through the chain.

In [ ]:
# Supplied: for the samples in `watch`, show the states x_t and the network's clean estimates.
@torch.no_grad()
def show_sampling(model, states, results, watch):
    rows, labels = [], []
    for i in watch:
        estimates = []
        for step, x in states.items():
            t = torch.full((1,), step)
            estimate = predict_clean(x[i:i + 1], t, model(x[i:i + 1], t), schedule)
            estimates.append(estimate.clamp(-1, 1))
        rows.append(torch.cat([x[i:i + 1] for x in states.values()] + [results[i:i + 1]]))
        rows.append(torch.cat(estimates))
        labels += [f"Sample {i}\nstate", "clean\nestimate"]
    show_rows(rows, labels=labels, titles=[f"t = {step}" for step in states] + ["result"])

first_samples = None
if sampler_ready:
    # 64 starting-noise images. Every later experiment reuses them.
    start = torch.randn(64, 1, 16, 16, generator=torch.Generator().manual_seed(37))
    shown_steps = [64, 48, 32, 16, 8, 1]
    first_samples, first_states = sample_ddpm(model, start, schedule, seed=31, record=shown_steps)
    first_updates = len(losses)
    row_labels = ["0–7", "8–15", "16–23", "24–31"]
    show_rows([first_samples[i:i + 8] for i in range(0, 32, 8)], labels=row_labels,
              suptitle=f"Generated after {first_updates} updates", size=0.8)
    watch = [0, 1, 2]  # change these numbers to follow other samples
    show_sampling(model, first_states, first_samples, watch)
else:
    print("Complete TODO 5 first, then rerun in order.")

**Inspect** the second figure. For each sample, the upper row shows the state
$x_t$ that the network receives; the lower row shows its clean estimate $\hat x_0$.

1. At which step can you first tell where the disk will be? Do you see it
   earlier in the state or in the clean estimate?
2. How does the clean estimate change from $t=48$ to $t=1$? Why is an early
   estimate blurry? Think of all the clean images that could hide in such a
   noisy state.
3. The estimate at $t=64$ is saturated speckle. Explain why with the factor
   printed in activity 5. Then use the weights at $t=64$ to explain why this
   does not ruin the sample.

_Your observations:_ …

### Diagnose before you tune: sampler or training?

Look at the 32 images from the short run and count those that are **empty**,
**faint**, or **not round**. Your counts will differ from your neighbours':
tiny numerical differences between computers grow during training.

There are two suspects: a mistake in the **sampler**, or a **network** that is
not trained well enough.

1. Which evidence do you already have about each suspect?
2. Put the number of one failed image into `watch` and rerun the cell above.
   At which step does the sample go wrong?

_Your diagnosis:_ …

Now test your diagnosis. The next cell trains for **1600 more updates** (1800
in total) and samples again with **the same random numbers**: the same starting
noise and the same fresh noise.

In [ ]:
samples = None
if first_samples is not None:
    if len(losses) == first_updates:  # rerunning this cell does not train again
        losses += train_updates(model, optimizer, train, schedule, 1600, train_generator)
    errors[f"Network, {len(losses)} updates"] = denoising_errors(model, valid, schedule)
    samples, states = sample_ddpm(model, start, schedule, seed=31, record=shown_steps)
    plot_errors(errors)
    print_errors(errors)
    for images, updates in [(first_samples, first_updates), (samples, len(losses))]:
        show_rows([images[i:i + 8] for i in range(0, 32, 8)], labels=row_labels,
                  suptitle=f"Generated after {updates} updates", size=0.8)
    # Optional: follow the same samples again with the better network.
    # show_sampling(model, states, samples, watch)
else:
    print("Generate images with the short run first, then rerun in order.")

**Record and explain:**

| Network | Empty or faint (of 32) | Not round (of 32) | Error at $t=1$ | Error at $t=32$ | Error at $t=64$ |
|---|---|---|---|---|---|
| 200 updates | … | … | … | … | … |
| 1800 updates | … | … | … | … | … |

1. Was your diagnosis right? Which observation made *before* the longer
   training already pointed to the network and not to the sampler?
2. Both figures use the same random numbers. Compare them image by image. What
   stays the same, what changes? What does the noise decide, and what does the
   network decide?
3. You have now turned one dial, **training quality**, while the sampler stayed
   fixed. Name a second dial that changes the *cost* of generating an image
   without retraining. (Activity 8 turns it.)

_Your observations:_ …

### What is the fresh noise for?

**Predict:** delete the fresh noise from every reverse step, so that each step
returns only the mean $\tilde\mu_t$. The 64 starting-noise images are still all
different. What will the 64 results look like?

_Your prediction:_ …

In [ ]:
if samples is not None:
    without_noise, _ = sample_ddpm(model, start, schedule, seed=31, fresh_noise=False)
    show_rows([samples[:8], without_noise[:8]], labels=["With fresh\nnoise", "Without"])
    for name, images in [("With fresh noise", samples), ("Without fresh noise", without_noise)]:
        spread = images.std(dim=0).mean().item()  # how much a pixel varies across the 64 images
        print(f"{name}: pixel standard deviation across the 64 images {spread:.3f}")
else:
    print("Train the network for the full 1800 updates first, then rerun in order.")

**Explain:** every result looks like a clean disk. Why is this sampler still a
poor generator? Where does the variety of DDPM samples come from? Look again at
the state weight and the fresh noise at $t=64$ in the plot above.

_Your explanation:_ …

## 7. Judge the generator

A generated image has no ground truth: there is no "correct" disk for a given
noise input. So we judge the **set** of samples. A good set has **fidelity**
(every sample looks like a real disk) and **diversity** (together the samples
cover the variety of the data), and it does not simply repeat the training set.

Two simple checks:

- **Copies?** For every generated image, find the nearest training image by
  Euclidean distance over the 256 pixels. As a yardstick, do the same for the
  validation images: they are new disks from the true distribution.
- **Same variety?** Compare the **bright area**, the fraction of pixels above 0
  and a stand-in for disk size, between generated and validation images.

**Predict:** if the network had memorised the training set, how would the
nearest-neighbour distances of generated images compare with those of
validation images?

_Your prediction:_ …

In [ ]:
# Supplied diagnostics.
def bright_area(images):
    return (images > 0).float().mean(dim=(1, 2, 3))  # fraction of pixels brighter than mid-gray

def nearest_training_image(images):
    distances = torch.cdist(images.flatten(1), train.flatten(1))  # Euclidean, over 256 pixels
    return distances.min(dim=1)  # distance to, and index of, the nearest training image

if samples is not None:
    generated_distance, generated_index = nearest_training_image(samples)
    validation_distance, _ = nearest_training_image(valid)
    print("Distance to the nearest training image")
    for name, d in [("Generated", generated_distance), ("Validation", validation_distance)]:
        print(f"  {name:10s} smallest {d.min().item():.2f} | median {d.median().item():.2f} | "
              f"largest {d.max().item():.2f}")
    closest = generated_distance.argsort()[:8]
    show_rows([samples[closest], train[generated_index[closest]]],
              labels=["Generated", "Nearest\ntraining"],
              titles=[f"{generated_distance[i].item():.2f}" for i in closest],
              suptitle="The eight generated images closest to a training image (distance above)")

    groups = [("Validation images", valid, GRAY),
              (f"Generated, {first_updates} updates", first_samples, BLUE),
              (f"Generated, {len(losses)} updates", samples, GREEN)]
    fig, axes = plt.subplots(len(groups), 1, figsize=(6, 4.6), sharex=True, sharey=True)
    for ax, (name, images, color) in zip(axes, groups):
        area = bright_area(images)
        empty = (area == 0).sum().item()
        ax.hist(area.clamp(max=0.399).numpy(), bins=np.linspace(0, 0.4, 21), color=color,
                rwidth=0.85)
        ax.set_title(f"{name}: mean {area.mean().item():.3f}, {empty} of {len(area)} empty",
                     fontsize=9, loc="left")
        ax.set_ylabel("Images")
        ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
        ax.set_axisbelow(True)
    axes[-1].set_xlabel("Bright area (fraction of pixels above 0)")
    fig.tight_layout()
    plt.show()
else:
    print("Train the network for the full 1800 updates first, then rerun in order.")

**Record and explain:**

1. Are the generated disks copies of training images? Compare the two distance
   summaries and look at the closest pairs.
2. Does the generated size distribution match the validation distribution?
   Compare with another pair: the direction of a mismatch can differ between
   computers. What does that say about judging a generator from one training run?
3. Every check is blind to some failures. Name one failure that the
   nearest-neighbour check would miss. Which experiment today produced
   perfect-looking images that fail as a *set*?

_Your observations:_ …

## 8. Try: a tempting shortcut

You have turned one dial, **training quality**. Now turn the second one: the
**cost of sampling**. From here on the network stays frozen, and only the
sampler changes.

Your DDPM sampler visits every timestep: 64, 63, …, 1. Each visit calls the
network once. The number of these **network evaluations** per batch is the cost
of sampling. The original DDPM sampler needed 1000 of them for every image.

The obvious way to save work: visit only every fourth timestep, 64, 60, 56, …,
4, and leave the DDPM step as it is. That makes 16 evaluations instead of 64.

**Predict:** what will the results look like? Clean disks, slightly worse
disks, or something else? Give a reason.

_Your prediction:_ …

In [ ]:
# Supplied: two more diagnostics.
def roughness(images):
    # Mean brightness difference between neighbouring pixels, with black = 0 and white = 1.
    across = (images[..., :, 1:] - images[..., :, :-1]).abs().mean(dim=(1, 2, 3))
    down = (images[..., 1:, :] - images[..., :-1, :]).abs().mean(dim=(1, 2, 3))
    return (across + down) / 4

def variety(images):
    return images.std(dim=0).mean().item()  # how much a pixel varies across the images of a batch

# The DDPM loop, but only at every `stride`-th timestep. The step itself is unchanged.
@torch.no_grad()
def sample_shortcut(model, initial, schedule, seed, stride=4):
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x = initial.clone()
    for step in range(schedule.steps, 0, -stride):  # 64, 60, 56, ..., 4
        z = torch.randn(x.shape, generator=generator)
        x = ddpm_step(model, x, step, z, schedule)
    return x

ddpm_ready = samples is not None
if ddpm_ready:
    model.requires_grad_(False)  # frozen: no experiment below changes the weights
    frozen = {name: value.clone() for name, value in model.state_dict().items()}
    reference = samples  # your DDPM results after 1800 updates: 64 evaluations per batch
    shortcut = sample_shortcut(model, start, schedule, seed=31)
    show_rows([reference[:8], shortcut[:8]],
              labels=["DDPM, 64\nevaluations", "Shortcut, 16\nevaluations"])
    for name, images in [("Validation images", valid), ("DDPM, 64 evaluations", reference),
                         ("Shortcut, 16 evaluations", shortcut)]:
        print(f"{name:25s} pixel values from {images.min().item():+.2f} to "
              f"{images.max().item():+.2f} | roughness {roughness(images).mean().item():.3f}")
else:
    print("Complete TODO 5 and activities 5 and 6 first, then rerun in order.")

### Whose fault: the network's or the loop's?

Both rows come from the same network and the same starting noise. In activity
6 you separated these two suspects with the **oracle**: a perfect denoiser for
one known image. A correct loop then leads every start to that image.

The oracle also keeps a **logbook**. At every call it notes the label $t$ it
receives and measures how much noise the state really contains. It can do that
because it knows the clean image: whatever the clean image does not explain is
noise.

Compare with what the label stands for. In training, every image with label
$t$ contained noise with standard deviation $\sqrt{1-\bar\alpha_t}$.

In [ ]:
# Supplied: run a sampler with the oracle, then print or plot the oracle's logbook.
def test_with_oracle(run):
    # `run(model)` runs a sampler on the 64 starts. Returns the oracle with its logbook.
    oracle = Oracle(valid[:1], schedule)
    result = run(oracle)
    # Largest pixel difference from the known image: 0 if every start ends at that image.
    oracle.deviation = (result - oracle.clean).abs().max().item()
    oracle.left = oracle.noise_in(result)  # the noise left in the results
    return oracle

def print_logbook(oracle):
    print(f"{'Label':>8s}{'Noise the label stands for':>30s}{'Noise in the state':>22s}")
    for label, noise in oracle.log:
        print(f"{label:8d}{(1 - schedule.abar[label]).sqrt().item():30.2f}{noise:22.2f}")
    print(f"{'result':>8s}{0:30.2f}{oracle.left:22.2f}")

# Colour, marker and marker size of every sampler that enters the logbook plot today.
STYLE = {"DDPM, 64 evaluations": (BLUE, "o", 3.5), "Shortcut, 16 evaluations": (ORANGE, "s", 5),
         "DDIM, 16 evaluations": (GREEN, "^", 6.5), "DDPM without fresh noise": (PURPLE, "D", 4)}

def plot_logbooks(names):
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    ax.plot(np.arange(T + 1), (1 - schedule.abar).sqrt().numpy(), color=GRAY, linewidth=1.5,
            label="What the label stands for")
    for name in names:
        color, marker, size = STYLE[name]
        labels, noise = zip(*logbooks[name].log)
        ax.plot(labels, noise, marker, color=color, markersize=size, markeredgecolor="white",
                markeredgewidth=0.5, label=f"State: {name}")
    ax.set(xlabel="Label $t$ given to the network   (sampling runs from right to left)",
           ylabel="Noise (standard deviation)", xlim=(0, T + 1), ylim=(-0.04, 1.08))
    ax.grid(color="#e6e6e6", linewidth=0.8)
    ax.legend(frameon=False, fontsize=9, loc="center right")
    fig.tight_layout()
    plt.show()

logbooks = {}
if ddpm_ready:
    logbooks["DDPM, 64 evaluations"] = test_with_oracle(
        lambda m: sample_ddpm(m, start, schedule, 31)[0])
    logbooks["Shortcut, 16 evaluations"] = test_with_oracle(
        lambda m: sample_shortcut(m, start, schedule, 31))
    for name, tested in logbooks.items():
        print(f"Oracle test of {name}: the results are up to {tested.deviation:.2f} away "
              "from the known image.")
    print("\nLogbook of the shortcut:")
    print_logbook(logbooks["Shortcut, 16 evaluations"])
    plot_logbooks(["DDPM, 64 evaluations", "Shortcut, 16 evaluations"])
else:
    print("Complete TODO 5 and activities 5 and 6 first, then rerun in order.")

**Diagnose:**

1. The oracle test clears one suspect. Which one, and why?
2. The first step runs at label 64. Which level does the state have afterwards?
   Which label does the loop give it next?
3. Read the last rows of the logbook. How much noise does a state at level 4
   contain? How much does the shortcut's state contain when it arrives there?

_Your diagnosis:_ …

> **The rule: a state and its label must agree.** A state with label $t$ has to
> contain the noise of level $t$, with standard deviation
> $\sqrt{1-\bar\alpha_t}$, as in training. Otherwise the network misjudges how
> much of its input is noise. A sampler may visit as few labels as it likes, as
> long as every step **lands on the level of the next label**.

## 9. Understand: a step that can jump

A DDPM step goes down exactly one level. We need a step from level $t$ to
**any** earlier level $s<t$.

**Split the state.** At label $t$ the network predicts the noise, which gives
the clean estimate $\hat x_0$ (activity 3). As in the DDPM step, we clip
$\hat x_0$ to $[-1,1]$. The clean estimate and a matching noise estimate
$\hat\epsilon$ are the two parts of the state:

$$x_t=\sqrt{\bar\alpha_t}\;\hat x_0+\sqrt{1-\bar\alpha_t}\;\hat\epsilon
\qquad\Longleftrightarrow\qquad
\hat\epsilon=\frac{x_t-\sqrt{\bar\alpha_t}\;\hat x_0}{\sqrt{1-\bar\alpha_t}}.$$

**Put it together again at level $s$.** The direct noising equation says what a
state at level $s$ looks like. Use it with the two estimates:

$$\boxed{\,x_s=\sqrt{\bar\alpha_s}\;\hat x_0+\sqrt{1-\bar\alpha_s}\;\hat\epsilon\,}$$

This is the step of the **DDIM** sampler (denoising diffusion implicit models).
It adds no fresh noise: the same state always leads to the same next state.

**Pen and paper:**

1. Set $s=t$. What does the step return?
2. Set $s=0$, where $\bar\alpha_0=1$. What does the step return?
3. A sampler with four evaluations visits the labels 64, 43, 22 and 1. On which
   level must each of its four steps land?

_Your answers:_ …

In [ ]:
@torch.no_grad()
def split_state(model, x, step, schedule):
    # The two parts of a state with label t (`step`): a clean estimate and the matching noise.
    t = torch.full((len(x),), step)
    a_t = schedule.abar[step]  # signal level at t
    clean = predict_clean(x, t, model(x, t), schedule).clamp(-1, 1)  # clipped estimate of x_0
    # TODO 6a: the noise estimate that goes with `clean`. Solve
    # x = sqrt(a_t) * clean + sqrt(1 - a_t) * noise for the noise.
    noise = None
    return clean, noise

def join_state(clean, noise, level, schedule):
    # A state at level s (`level`), put together from a clean image and noise with variance 1.
    a_s = schedule.abar[level]  # signal level at s
    # TODO 6b: weight `clean` and `noise` as the direct noising equation does at level s.
    return None

@torch.no_grad()
def ddim_step(model, x, step, earlier, schedule):
    # Jump from label t (`step`) to an earlier level s (`earlier`), without fresh noise.
    clean, noise = split_state(model, x, step, schedule)
    if noise is None:  # split_state is unfinished
        return None
    return join_state(clean, noise, earlier, schedule)

# Check with the oracle: build a state at level 32 from a known image and known noise.
known = valid[:1]
check_noise = torch.randn(4, 1, 16, 16, generator=torch.Generator().manual_seed(5))
at_32 = q_sample(known.repeat(4, 1, 1, 1), torch.full((4,), 32), check_noise, schedule)
at_10 = q_sample(known.repeat(4, 1, 1, 1), torch.full((4,), 10), check_noise, schedule)
checker = Oracle(known, schedule)
split_ready = split_state(checker, at_32, 32, schedule)[1] is not None
join_ready = join_state(known, check_noise, 10, schedule) is not None
step_ready = split_ready and join_ready
if split_ready:
    calls = len(checker.log)
    found = split_state(checker, at_32, 32, schedule)[1]
    assert len(checker.log) == calls + 1, "Call the network only once per step: reuse `clean`."
    assert torch.allclose(found, check_noise, atol=1e-4), \
        "TODO 6a: with the oracle, the noise part must be exactly the noise that was added."
if join_ready:
    assert torch.allclose(join_state(known, check_noise, 10, schedule), at_10, atol=1e-5), \
        "TODO 6b: joined at level 10, image and noise must give the image noised to level 10."
if step_ready:
    stay = ddim_step(checker, at_32, 32, 32, schedule)
    assert torch.allclose(stay, at_32, atol=1e-5), "With s = t the state must not change."
    finish = ddim_step(checker, at_32, 32, 0, schedule)
    assert torch.allclose(finish, known.expand(4, -1, -1, -1), atol=1e-5), \
        "With s = 0 the step must return the clean estimate."
    assert torch.allclose(ddim_step(checker, at_32, 32, 10, schedule), at_10, atol=1e-5), \
        "A jump from 32 to 10 must give the same image and the same noise at level 10."
    print("Jump checked: s = t changes nothing, s = 0 returns the clean estimate, "
          "and a jump from 32 lands exactly on level 10.")
else:
    missing = [name for name, ready in (("TODO 6a", split_ready), ("TODO 6b", join_ready))
               if not ready]
    print(f"Complete {' and '.join(missing)}, then rerun this cell.")

### Plan the jumps

A sampler with $K$ evaluations visits $K$ labels, evenly spaced from $T$ down
to 1. Every step must land on the level of the **next** label; the last step
lands on level 0, the clean image.

In [ ]:
def jump_plan(steps, evaluations):
    # The jumps (t, s) of a sampler: from every label t that it visits to the level s it lands on.
    labels = torch.linspace(steps, 1, evaluations).round().long().tolist()  # from T down to 1
    # TODO 7: the landing level of every jump: the next label in the list,
    # and the clean level 0 after the last label. Build a list as long as `labels`.
    landings = None
    if landings is None:
        return None
    return list(zip(labels, landings))

@torch.no_grad()
def sample_ddim(model, initial, schedule, evaluations):
    # One network evaluation per jump. The only randomness is the starting noise `initial`.
    model.eval()
    x = initial.clone()
    for step, earlier in jump_plan(schedule.steps, evaluations):
        x = ddim_step(model, x, step, earlier, schedule)
    return x

plan_ready = jump_plan(T, 4) is not None
if plan_ready:
    message = "Every jump must land on the label that the next jump starts from; the last on 0."
    assert jump_plan(T, 4) == [(64, 43), (43, 22), (22, 1), (1, 0)], message
    assert jump_plan(T, 1) == [(64, 0)], "A single jump goes from T straight to level 0."
    assert jump_plan(T, T) == [(t, t - 1) for t in range(T, 0, -1)], message
    for evaluations in (4, 8):
        jumps = "  ".join(f"{t}→{s}" for t, s in jump_plan(T, evaluations))
        print(f"{evaluations} evaluations: {jumps}")
else:
    print("Complete TODO 7, then rerun this cell.")
ddim_ready = step_ready and plan_ready and ddpm_ready

### Verify before you trust pictures

The shortcut also returned images of the right shape with finite values. So
test the new sampler as you tested the old ones: with the oracle. It must lead
every start to the known image, call the network exactly as often as planned,
and keep the noise in the state at the level of the label, at every call.

In [ ]:
if ddim_ready:
    for evaluations in (64, 16, 4, 1):
        oracle = test_with_oracle(lambda m: sample_ddim(m, start, schedule, evaluations))
        gap = max(abs(noise - (1 - schedule.abar[label]).sqrt().item())
                  for label, noise in oracle.log)
        assert len(oracle.log) == evaluations, "One network evaluation per planned jump."
        assert gap < 0.03, "At some call the noise in the state does not fit its label."
        # One jump from t = 64 divides by a tiny signal weight, which magnifies rounding errors.
        assert oracle.deviation < 2e-3, "Every start must end at the known image."
        print(f"{evaluations:2d} evaluations: results up to {oracle.deviation:.0e} away from the "
              f"known image | largest gap between state and label {gap:.3f}")
    logbooks["DDIM, 16 evaluations"] = test_with_oracle(
        lambda m: sample_ddim(m, start, schedule, 16))
    plot_logbooks(["Shortcut, 16 evaluations", "DDIM, 16 evaluations"])
else:
    print("Complete TODOs 5 to 7, then rerun in order.")

**Explain:**

1. Shortcut and DDIM both call the network 16 times. Where do their markers lie
   in the plot? What does the DDIM step do that the shortcut does not?
2. With the oracle, a single evaluation is enough to recover the image. A
   trained network cannot generate a disk in one evaluation. Why not? Think of
   the clean estimate at $t=64$ that you inspected in activity 6, and of what the
   best possible denoiser could say about pure noise.

_Your explanation:_ …

## 10. Experiment: how few evaluations are enough?

You now have a sampler that is correct for any number of evaluations. Whether
its images stay good is an empirical question. Plan the comparison before you
look at pictures:

| | |
|---|---|
| Question | How does the number of network evaluations change the generated images? |
| Changed | DDIM evaluations: 64, 16, 4 |
| Kept fixed | the frozen network, the 64-level schedule, the 64 starting-noise images, the DDIM step |
| Cost | network evaluations per batch, and seconds |
| Evidence | the first 16 images of every run, unselected; bright area; roughness; change per image |

DDIM adds no fresh noise, so each starting noise leads to exactly one image.
That allows a **paired** comparison: how far does an image move when only the
budget changes?

**Predict:** which change do you expect first as the budget falls: speckle,
disks of a different size, less variety, or little visible change?

_Your prediction:_ …

In [ ]:
# Supplied: how far is every image from its partner, the image made from the same starting
# noise? Euclidean distance over the 256 pixels, one per pair; the median of these distances.
def paired_change(images, partners):
    return (images - partners).flatten(1).norm(dim=1).median().item()

budgets = (64, 16, 4)  # after the first run, add more budgets, for example 8, 2 and 1

def describe(name, images, evaluations="", seconds="", change=""):
    area = bright_area(images)
    spread = area.std(unbiased=False).item()
    print(f"{name:15s}{evaluations:>12}{seconds:>9}{area.mean().item():10.3f} ± {spread:.3f}"
          f"{(area == 0).sum().item():7d}{roughness(images).mean().item():11.3f}{change:>8}")

if ddim_ready:
    by_budget, seconds = {}, {}
    for evaluations in budgets:
        began = time.perf_counter()
        by_budget[evaluations] = sample_ddim(model, start, schedule, evaluations)
        seconds[evaluations] = time.perf_counter() - began
    largest = by_budget[max(budgets)]
    print(f"{'Images':15s}{'Evaluations':>12s}{'Seconds':>9s}{'Bright area':>18s}"
          f"{'Empty':>7s}{'Roughness':>11s}{'Change':>8s}")
    describe("Validation", valid)
    describe("DDPM", reference, T)
    for evaluations in budgets:
        change = paired_change(by_budget[evaluations], largest)
        describe("DDIM", by_budget[evaluations], evaluations, f"{seconds[evaluations]:.2f}",
                 f"{change:.2f}")
    unrelated = paired_change(largest, largest.roll(1, dims=0))
    print(f"\nChange: median distance to the image that {max(budgets)} evaluations make from "
          f"the same start.\nFor scale: two unrelated images of that run are about "
          f"{unrelated:.1f} apart.")
    show_rows([by_budget[evaluations][:16] for evaluations in budgets], size=0.6,
              labels=[f"{evaluations}\nevaluations" for evaluations in budgets],
              suptitle="DDIM: every column starts from the same noise")
    assert all(torch.equal(value, frozen[name]) for name, value in model.state_dict().items())
    print("The weights are unchanged: every difference between the rows comes from the sampler.")
else:
    print("Complete TODOs 5 to 7, then rerun in order.")

**Record and interpret:**

1. **Gain.** How much cheaper are 16 evaluations than 64, in network
   evaluations and in seconds?
2. **Change.** Compare the rows column by column, then read the *Change*
   column. Is the change at 16 evaluations large or small on the scale printed
   below the table? And at 4?
3. **Find the breaking point.** Add 8, 2 and 1 to `budgets` and rerun. What is
   the smallest budget you would still accept? Then explain the failure at the
   smallest budgets: print `jump_plan(T, 2)`. Which clean estimate does the
   first jump trust almost completely? You inspected it in activity 6.
4. **Limits.** Our disks are a very simple family of images. Why might
   photographs need more evaluations? What do 64 images from one training run
   allow you to claim?

_Your record:_ …

## 11. Individual exit ticket

Answer for yourself, in four sentences:

1. What does one DDPM step do with the network's prediction?
2. What must be true of a state and its label, and why?
3. How does the DDIM step make sure of it for a jump of any size?
4. What did you keep fixed when you compared sampling budgets, and what does
   the comparison **not** show?

_Your exit ticket:_ …

## 12. Extended exercise: old noise or fresh noise?

**A puzzle.** In activity 6 you deleted the fresh noise from the DDPM step, and
all 64 results came out as nearly the same disk. The DDIM step adds no fresh
noise either, yet its results are as varied as before. The next cell shows both
and enters the stripped-down DDPM sampler into the oracle's logbook.

**Before running:** both samplers are deterministic. What does each of them do
with the noise that is already in the state? Use the rule from activity 8.

_Your explanation:_ …

In [ ]:
if ddim_ready:
    ddim_64 = sample_ddim(model, start, schedule, 64)
    show_rows([without_noise[:8], ddim_64[:8]],
              labels=["DDPM without\nfresh noise", "DDIM, 64\nevaluations"])
    for name, images in [("Validation images", valid), ("DDPM without fresh noise", without_noise),
                         ("DDIM, 64 evaluations", ddim_64)]:
        print(f"{name:25s} variety (pixel standard deviation across the images) "
              f"{variety(images):.3f}")
    logbooks["DDPM without fresh noise"] = test_with_oracle(
        lambda m: sample_ddpm(m, start, schedule, 31, fresh_noise=False)[0])
    plot_logbooks(["DDPM, 64 evaluations", "DDPM without fresh noise", "DDIM, 16 evaluations"])
else:
    print("Complete TODOs 5 to 7 first.")

**What the plot shows.** A state at level $s$ needs noise with variance
$1-\bar\alpha_s$. Call this its **noise budget**.

- The DDPM step keeps only part of the old noise and fills the rest of the
  budget with fresh noise. Delete the fresh noise and the state is far too
  clean for its label: the opposite of the shortcut's mistake, and the same
  broken rule.
- The DDIM step fills the whole budget with old noise, rescaled to level $s$.

So the budget can be filled from two sources, and in any mixture. Replace
the old noise $\hat\epsilon$ in the DDIM step by a mixture of old noise and
fresh noise $z$:

$$x_s=\sqrt{\bar\alpha_s}\;\hat x_0+\sqrt{1-\bar\alpha_s}\;
\big(u_{\text{old}}\;\hat\epsilon+u_{\text{fresh}}\;z\big),
\qquad z\sim\mathcal N(0,I).$$

**Pen and paper:** $\hat\epsilon$ and $z$ each have variance 1, and the
mixture must have variance 1 as well. Independent noise terms add their
variances, and a weight $u$ multiplies a variance by $u^2$ (activity 2). Which weights $u_{\text{old}}$ and $u_{\text{fresh}}$ give the
mixture variance 1, with a **share** $\lambda$ of it coming from the fresh
noise? Check your result: which step do you get for $\lambda=0$?

_Your weights:_ …

**DDPM is one setting of this dial.** For a step from $t$ to $s$, its share is
$\lambda=\dfrac{1-\bar\alpha_t/\bar\alpha_s}{1-\bar\alpha_t}$. The cell does
not derive this. It checks it: with this share your new step must reproduce
your `ddpm_step`.

In [ ]:
@torch.no_grad()
def mixed_step(model, x, step, earlier, share, z, schedule):
    # Like ddim_step, but a `share` (0 to 1) of the noise in the new state is the fresh noise `z`.
    clean, old = split_state(model, x, step, schedule)
    # TODO 8: mix the old noise `old` and the fresh noise `z`. The mixture needs variance 1,
    # with a `share` of that variance coming from `z`. `share` is a tensor: use share.sqrt().
    noise = None
    if noise is None:
        return None
    return join_state(clean, noise, earlier, schedule)

# Supplied: the share of fresh noise that turns a jump from t to s into a DDPM step.
def ddpm_share(step, earlier, schedule):
    a_t, a_s = schedule.abar[step], schedule.abar[earlier]
    return ((1 - a_t / a_s) / (1 - a_t)).clamp(0, 1)

# Supplied: `share` is a number between 0 and 1, or "ddpm" for DDPM's share in every step.
@torch.no_grad()
def sample_mixed(model, initial, schedule, evaluations, share, seed):
    model.eval()
    generator = torch.Generator().manual_seed(seed)
    x = initial.clone()
    for step, earlier in jump_plan(schedule.steps, evaluations):
        z = torch.randn(x.shape, generator=generator)
        fresh = ddpm_share(step, earlier, schedule) if share == "ddpm" else torch.tensor(share)
        x = mixed_step(model, x, step, earlier, fresh, z, schedule)
    return x

mix_ready = False
if ddim_ready:
    state = q_sample(valid[:4], torch.full((4,), 32), check_noise, schedule)
    z = torch.randn(4, 1, 16, 16, generator=torch.Generator().manual_seed(6))
    mix_ready = mixed_step(model, state, 32, 20, torch.tensor(0.5), z, schedule) is not None
if mix_ready:
    a_20, half, no_z = schedule.abar[20], torch.tensor(0.5), torch.zeros_like(z)
    ddim_20 = ddim_step(model, state, 32, 20, schedule)
    old_only = mixed_step(model, state, 32, 20, torch.tensor(0.0), z, schedule)
    assert torch.allclose(old_only, ddim_20, atol=1e-5), \
        "With share 0 the step must equal your DDIM step."
    # Half fresh: old and fresh noise get the same weight, sqrt(0.5).
    with_z = mixed_step(model, state, 32, 20, half, z, schedule)
    without_z = mixed_step(model, state, 32, 20, half, no_z, schedule)
    assert torch.allclose(with_z - without_z, (0.5 * (1 - a_20)).sqrt() * z, atol=1e-5), \
        "With share 0.5 the fresh noise needs the weight sqrt(0.5) in the mixture."
    clean_part = a_20.sqrt() * ddim_step(model, state, 32, 0, schedule)
    assert torch.allclose(without_z, clean_part + math.sqrt(0.5) * (ddim_20 - clean_part),
                          atol=1e-5), \
        "With share 0.5 the old noise needs the weight sqrt(0.5) in the mixture as well."
    as_ddpm = mixed_step(model, state, 32, 31, ddpm_share(32, 31, schedule), z, schedule)
    assert torch.allclose(as_ddpm, ddpm_step(model, state, 32, z, schedule), atol=1e-5), \
        "With DDPM's share, a step from 32 to 31 must equal ddpm_step."
    repeat = sample_mixed(model, start, schedule, T, "ddpm", seed=31)
    mismatch = (repeat - reference).abs().max().item()
    print("Mixed step checked: share 0 is your DDIM step, and DDPM's share reproduces the DDPM "
          f"sampler\n(largest pixel difference after all 64 steps: {mismatch:.0e}).")

    # The step at t = 1 lands on the clean level 0, whose noise budget is empty: start at t = 2.
    shares = {t: ddpm_share(t, t - 1, schedule).item() for t in range(2, T + 1)}
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(list(shares), list(shares.values()), color=BLUE)
    ax.set(xlabel="Timestep $t$   (sampling runs from right to left)",
           ylabel="Share of fresh noise", xlim=(1, T), ylim=(0, 1.03))
    ax.set_title("How much of the noise does one DDPM step replace?", fontsize=10)
    ax.grid(color="#e6e6e6", linewidth=0.8)
    fig.tight_layout()
    plt.show()
    print("Share of fresh noise in the DDPM step at t = 64, 63, 48, 32, 16, 2:",
          "  ".join(f"{shares[t]:.2f}" for t in (64, 63, 48, 32, 16, 2)))
elif ddim_ready:
    print("Complete TODO 8, then rerun this cell.")
else:
    print("Complete TODOs 5 to 7 first.")

**Read the plot.** In which part of the chain does a DDPM step keep most of the
old noise? Where does it replace nearly all of it?

**Predict.** The next cell runs three samplers with 64 evaluations each: old
noise only ($\lambda=0$, your DDIM sampler), DDPM's share, and fresh noise only
($\lambda=1$: rebuild the state from the clean estimate and all-new noise at
every step). Every sampler runs three times: once as before, once with
different fresh noise, once with different starting noise.

1. Which of the three samplers will produce disks?
2. For each sampler: which of the two changes will alter its images?

_Your reading and prediction:_ …

In [ ]:
if mix_ready:
    other_start = torch.randn(64, 1, 16, 16, generator=torch.Generator().manual_seed(38))
    blank = torch.full((1, 1, 16, 16), float("nan"))  # an empty square between the groups
    runs = {"As before": [], "Other fresh noise": [], "Other starting noise": []}
    print(f"{'Noise in every step':24s}{'Bright area':>16s}{'Empty':>7s}{'Variety':>9s}"
          f"{'Change with other':>20s}{'Change with other':>20s}")
    print(f"{'':56s}{'fresh noise':>20s}{'starting noise':>20s}")
    for name, share in [("Old only (DDIM)", 0.0), ("DDPM's share", "ddpm"), ("Fresh only", 1.0)]:
        before = sample_mixed(model, start, schedule, 64, share, seed=31)
        fresh = sample_mixed(model, start, schedule, 64, share, seed=32)       # other fresh noise
        begun = sample_mixed(model, other_start, schedule, 64, share, seed=31)  # other start
        area = bright_area(before)
        print(f"{name:24s}{area.mean().item():8.3f} ± {area.std(unbiased=False).item():.3f}"
              f"{(area == 0).sum().item():7d}{variety(before):9.3f}"
              f"{paired_change(before, fresh):20.2f}{paired_change(before, begun):20.2f}")
        for row, images in zip(runs.values(), (before, fresh, begun)):
            row += [images[:4], blank]
    titles = [""] * 12  # a title above the second image of each group of four
    titles[1], titles[6], titles[11] = "Old noise only", "DDPM's share", "Fresh noise only"
    show_rows([torch.cat(row[:-1]) for row in runs.values()], labels=list(runs), titles=titles,
              size=0.62)
else:
    print("Complete TODOs 5 to 8 first.")

**Explain:**

1. All three samplers produce disks. Which property do they share that the
   shortcut and the stripped-down sampler lack?
2. For each sampler, what decides which disk appears: the starting noise or the
   fresh noise? Explain the row of DDPM with its share of fresh noise at
   $t=64$.
3. You want to (a) compare budgets image by image, as in activity 10; (b) store
   an image as its starting noise and recreate it later; (c) draw several
   different images from one starting noise. Which sampler fits each wish?
4. Compare the *Variety* column with the validation images (the value is
   printed in the puzzle cell above). Which sampler has the lowest variety?
   Suggest a reason: what kind of image is the clean estimate while the state
   is still very noisy?

_Your observations:_ …

## 13. Extended exercise: ask for a disk on one side

So far the noise decided where a disk appears. Text-to-image models let the
user decide: they **condition** the network on a prompt. Our "prompt" is as
simple as possible: should the disk lie in the **left** or the **right** half
of the image?

**One more input.** Every training image gets a label, LEFT or RIGHT, read off
the horizontal position of its disk. The network receives the label as a third
input: one learned vector per label, added to the time embedding, so that it
reaches every block. For one training image in five the label is hidden and
replaced by NO_LABEL. One network thereby learns two predictions: the noise
given the label, $\epsilon_\theta(x_t,t,c)$, and the noise without a label,
$\epsilon_\theta(x_t,t,\varnothing)$.

The next cell trains this second network for 1500 updates, about 20 seconds.

In [ ]:
LEFT, RIGHT, NO_LABEL = 0, 1, 2

def horizontal_position(images):
    # Brightness-weighted mean column of every image, from -1 (left edge) to +1 (right edge).
    brightness = (images[:, 0] + 1) / 2
    columns = torch.linspace(-1, 1, 16)
    return (brightness.sum(dim=1) * columns).sum(dim=1) / brightness.sum(dim=(1, 2)).clamp(min=1e-6)

class LabelUNet(TinyTimeUNet):
    # The network from Session 1 with one more input: the side on which the disk should lie.
    def __init__(self, steps):
        super().__init__(steps)
        self.label = nn.Embedding(3, 32)  # one vector each for LEFT, RIGHT and NO_LABEL

    def forward(self, x, t, label):
        emb = self.embedding(time_features(t, self.steps)) + self.label(label)  # (N, 32)
        return self.unet(x, emb)

def train_with_labels(network, images, sides, schedule, updates, generator):
    optimizer = torch.optim.Adam(network.parameters(), lr=0.002)
    network.train()
    for _ in range(updates):
        index = torch.randint(len(images), (32,), generator=generator)
        label = sides[index].clone()
        label[torch.rand(32, generator=generator) < 0.2] = NO_LABEL  # hide one label in five
        t = torch.randint(1, schedule.steps + 1, (32,), generator=generator)
        noise = torch.randn(32, 1, 16, 16, generator=generator)
        prediction = network(q_sample(images[index], t, noise, schedule), t, label)
        loss = F.mse_loss(prediction, noise)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return network.requires_grad_(False)

conditional = None
if ddim_ready:
    sides = (horizontal_position(train) > 0).long()  # the label of every training image
    print(f"Training images labelled RIGHT: {sides.float().mean().item():.0%}")
    torch.manual_seed(17)
    began = time.perf_counter()
    conditional = train_with_labels(LabelUNet(T), train, sides, schedule, 1500,
                                    torch.Generator().manual_seed(23))
    print(f"Trained the network with a label input in {time.perf_counter() - began:.0f} s.")
else:
    print("Complete TODOs 5 to 7 first: this activity samples with your DDIM sampler.")

**Guidance.** The network can now be asked for a side. **Classifier-free
guidance** lets you choose how much the label counts. At every step it takes
the direction from the prediction without a label to the prediction with the
label, and goes $w$ times as far:

$$\tilde\epsilon=\epsilon_\theta(x_t,t,\varnothing)
+w\,\big(\epsilon_\theta(x_t,t,c)-\epsilon_\theta(x_t,t,\varnothing)\big).$$

The **guidance weight** $w$ is chosen at sampling time. No new sampler is
needed: the guided prediction $\tilde\epsilon$ takes the place of the network
in your DDIM sampler.

**Predict:**

1. Which prediction does the formula return for $w=0$, and which for $w=1$?
2. For $w=0$, $1$ and $3$: roughly which fraction of the disks will land on
   the requested side?
3. What could a large weight cost? Think of the variety of the positions.
4. How many network evaluations does a batch need with 16 DDIM jumps?

_Your prediction:_ …

In [ ]:
class Guided(nn.Module):
    # Presents the network with a label input to a sampler as an ordinary denoiser model(x, t).
    def __init__(self, network, side, weight):
        super().__init__()
        self.network, self.side, self.weight, self.calls = network, side, weight, 0

    def forward(self, x, t):
        plain = self.network(x, t, torch.full((len(x),), NO_LABEL))      # without a label
        labelled = self.network(x, t, torch.full((len(x),), self.side))  # for the requested side
        self.calls += 2
        # TODO 9: start from `plain` and go `self.weight` times the way towards `labelled`.
        return plain

weights = (0, 1, 3)  # after the first run, try a larger weight as well, for example 8

guidance_ready = False
if conditional is not None:
    probe_t = torch.full((4,), 32)
    probe_x = q_sample(valid[:4], probe_t, check_noise, schedule)
    plain = conditional(probe_x, probe_t, torch.full((4,), NO_LABEL))
    labelled = conditional(probe_x, probe_t, torch.full((4,), LEFT))
    guidance_ready = not torch.allclose(Guided(conditional, LEFT, 3)(probe_x, probe_t), plain)
if guidance_ready:
    for weight in (0, 1, 3):
        guided = Guided(conditional, LEFT, weight)(probe_x, probe_t)
        assert torch.allclose(guided, plain + weight * (labelled - plain), atol=1e-5), \
            f"Wrong for weight {weight}: weight 0 must give `plain`, weight 1 `labelled`."
    print(f"{'Weight':>6s}{'Requested':>11s}{'On that side':>14s}{'Spread of':>11s}"
          f"{'Bright area':>18s}{'Empty':>7s}{'Network':>9s}")
    print(f"{'':6s}{'side':>11s}{'':14s}{'position':>11s}{'':25s}{'calls':>9s}")
    rows, row_labels = [], []
    for weight in weights:
        requests = [(RIGHT, "none")] if weight == 0 else [(LEFT, "left"), (RIGHT, "right")]
        for side, name in requests:
            guided = Guided(conditional, side, weight)
            images = sample_ddim(guided, start, schedule, 16)
            area = bright_area(images)
            position = horizontal_position(images)[area > 0]  # empty images have no position
            on_side = ((position > 0) == (side == RIGHT)).float().mean().item()
            hit = f"{on_side:.0%}" + (" right" if weight == 0 else "")
            print(f"{weight:6g}{name:>11s}{hit:>14s}{position.std().item():11.3f}"
                  f"{area.mean().item():10.3f} ± {area.std(unbiased=False).item():.3f}"
                  f"{(area == 0).sum().item():7d}{guided.calls:9d}")
            rows.append(images[:12])
            row_labels.append(f"w = {weight:g}" + ("" if weight == 0 else f", {name}"))
    show_rows(rows, labels=row_labels, size=0.62,
              suptitle="Guided DDIM, 16 jumps: every column starts from the same noise")
elif conditional is not None:
    print("Guidance has no effect yet. Complete TODO 9, then rerun this cell.")
else:
    print("Complete TODOs 5 to 7 and run the previous cell first.")

The **spread of position** is the standard deviation of the horizontal
positions of the disks, a measure of their variety.

**Inspect and discuss:**

1. Every column starts from the same noise. What happens to a disk when only
   the label changes? What happens as $w$ grows?
2. Read the table. What does a larger weight buy, and what does it cost? Add
   a weight of 8 to `weights` and compare the images of a row with each other.
3. A fraction measured on 64 images is uncertain: near 50% its standard error
   is about 6 percentage points. Which differences in the table do you trust?
4. Guidance changed neither the weights nor the training. Which dial did it
   turn, and what does it cost per image? For which weight is one of the two
   network calls wasted?
5. A text-to-image model replaces our two labels by a prompt. Its network
   needs a vector that describes the prompt. Where in `LabelUNet` would that
   vector enter?

_Your observations:_ …

### Where next

- **Fill in a hidden region:** activity 8 of `diffusion.ipynb` turns your DDPM
  sampler into an inpainting tool.
- **Lab 7** in the course book repeats the comparison of sampling budgets with
  the book's own network and adds exercises: break the sampler and catch the
  error, sample from an undertrained network, and remove the time input.
- **Lesson 7** connects conditioning and guidance to latent diffusion and
  current image generators. Its additional material derives the reverse step
  and explains why a DDIM jump is consistent.

### References

- [Course book, Lesson 7: Diffusion Image Generation](https://healthml.github.io/deep-learning-book/chapters/07-diffusion/concepts.html), [Lab 7: How Many Denoising Steps Do We Need?](https://healthml.github.io/deep-learning-book/chapters/07-diffusion/lab.html), and the [additional material](https://healthml.github.io/deep-learning-book/chapters/07-diffusion/additional-material.html): notation, the reverse process, the shortcut, the DDIM comparison, and the guidance experiment that this notebook adapts.
- [Ho, Jain & Abbeel (2020), Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239): the noise-prediction loss and the DDPM sampler.
- [Nichol & Dhariwal (2021), Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672): the cosine schedule.
- [Song, Meng & Ermon (2021), Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502): the DDIM step, and the family of samplers between DDIM and DDPM used in activity 12.
- [Ho & Salimans (2022), Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598): guidance without a separate classifier.

Original classroom exercise with synthetic data. Repository licenses: code MIT;
original prose and exercises CC BY-SA 4.0.